In [ ]:
#| hide
from answerui.all import *

# answerui

> A semantic HTML UI framework for FastHTML apps

answerui styles semantic HTML for [FastHTML](https://fastht.ml) apps. A component is an HTML tag, a variant or a state is an attribute, and classes do only utility jobs, so page markup stays small and readable. Every component's CSS and behavior lives in one registry, and a page carries the whole registry as one stylesheet and one script. An app changes the look by registering ordinary CSS in the same registry, scoped to a token, a component, or a region.
answerui is built for a human working with an LLM in a persistent Python kernel. Its pyskill, `answerui.skill`, teaches the model the library: how a page is built, how to read what is registered, how to change it, and how to check the result.

## Install

```sh
pip install -e .
```

## Use

The registry's headers carry the stylesheet and the behavior script. They render per response, so a rule registered later is in the next page:

In [ ]:
from fasthtml.common import *
from answerui.all import *

In [ ]:
app = FastHTML(hdrs=reg.hdrs())
rt = app.route

`reg.serve(app)` serves the same assets from URLs named by a hash of their content, which the browser caches. They are rebuilt when a rule changes, and the next page links the new build as a new URL.

A page is native HTML inside layout containers, with a component wherever the platform has no element for the job. Variants and severities are attributes:

In [ ]:
page = Card(H2('Profile'), Field('Name', Input(name='name')), Cluster(Button('Save'), Button('Cancel', secondary=True)))
print(to_xml(page, Alert('Saved.', success=True, dismissible=True)))

<div stack card>
  <h2>Profile</h2>
  <div field>
<label>Name      <input name="name">
</label>  </div>
  <div cluster>
<button>Save</button><button secondary>Cancel</button>  </div>
</div>

<div card alert success>
  <div stack style="--gap: var(--space-2);">Saved.</div>
<button type="button" secondary dismiss>Dismiss</button></div>



Rendered with the registry's stylesheet. In a notebook, `demo(page)` shows the same thing live:

![](index_files/page.png)

## Customise

A change is ordinary CSS registered at the smallest scope that covers it: a token on the root for the whole app, a rule on a component, or a selector for a region. App rules sit in a cascade layer above the framework's, so nothing in the framework needs editing:

In [ ]:
reg.rule('theme', ':root { --hue: 150; --radius: 0; }')
Button.rule('caps', 'text-transform: uppercase;')

Button.caps [app] button, a[button]

A variant is a new treatment declared on the component. It becomes an attribute the constructor accepts and appears in the component's documentation:

In [ ]:
Button.variant('cta', '--pad-block: var(--space); --pad-inline: var(--space2); font-size: var(--text1);', doc='The landing page call to action.')

Button.cta [app] :is(button, a[button])[cta]: The landing page call to action.

![](index_files/custom.png)

In [ ]:
Button

### `Button(*children, **attrs)`

Raised at rest, down while active, busy, or expanded, latched by `aria-pressed`; `secondary` for the lesser action; a link carrying `button` is styled the same way.

Root selector: `button, a[button]`.

Declared variants:
- `secondary`: The lesser action: the neutral surface with a hairline border, and the accent wash when latched.
- `quiet`: No face, border, or shadow at rest, for an action that should not compete: dismiss, close, and toolbar buttons.
- `editable`: A value drawn as the dashed ghost of the well it becomes; click to edit, and the app swaps the form in.
- `cta`: The landing page call to action.

Rules:
- `base` (components)
- `icon-only` (components): A button whose only child is an svg is square; give it an aria-label.
- `caps` (app)

## Previews in notebooks

`demo(page)` renders markup in an iframe with the current stylesheet and behaviors. The notebook stores the frame, which holds the whole stylesheet, so a notebook with many previews is large, and every style change rewrites every preview.

To keep frames out of git while keeping them in your working copy, run once in each clone:

```sh
answerui-install-hooks
```

This adds a git clean filter that removes preview frames from notebooks as they are staged. The notebooks on disk keep every preview, and the file still records each preview's markup. A fresh clone shows previews after the cells run.

To regenerate previews when building docs with nbdev, name the processor in `pyproject.toml`:

```toml
[tool.nbdev]
doc_procs = ["answerui.previews:exec_previews"]
```

`nbdev-docs` then runs each notebook and renders the previews from the current code.

## With an LLM

`answerui.skill` is a [pyskill](https://github.com/AnswerDotAI/pyskills): a module whose docstring teaches a model the library. `list_pyskills()` finds it, `doc(answerui.skill)` is the guide, and `doc(Button)` describes a component, its variants, and its rules.